In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
    config
    )

In [5]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st', additional_kwargs={}, response_metadata={}, id='f23f9400-ac28-473f-875a-474d19acbc8b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 990, 'prompt_tokens': 2714, 'total_tokens': 3704, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 896, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ElcZndJSp0NjjoX4FwFIhFuudwyWb', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1ef6f-4b91-76a1-a3e2-fea8b9542261-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'San Francisco', 'flyTo': 'Tokyo', 'departureDate': '31/03/2027

In [6]:
print(response["messages"][-1].content)

Here are direct flights from San Francisco to Tokyo on March 31 (one-way), all nonstop.

| Route | Times (local) | Cabin | Price | Booking |
|---|---|---|---|---|
| SFO → NRT (ZIPAIR) | Mar 31 16:50 → Apr 01 20:00 (11h 10m) | Economy | 634 USD | https://kiwi.com/u/m2bfyq5qgz |
| SFO → NRT (Japan Airlines JL57) | Mar 31 13:30 → Apr 01 16:30 (11h 0m) | Economy | 3,523 USD | https://kiwi.com/u/jtkrrz353f |
| SFO → NRT (All Nippon Airways NH7) | Mar 31 12:20 → Apr 01 15:25 (11h 5m) | Economy | 829 USD | https://kiwi.com/u/r4bsdjydkg |
| SFO → HND (Japan Airlines JL1) | Mar 31 11:55 → Apr 01 15:05 (11h 10m) | Economy | 3,519 USD | https://kiwi.com/u/x6r4cuzkuv |
| SFO → HND (All Nippon Airways NH107) | Mar 31 01:45 → Apr 01 05:00 (11h 15m) | Economy | 1,184 USD | https://kiwi.com/u/spsvq9yujp |

Best price: ZIPAIR nonstop SFO → NRT for 634 USD.
Shortest duration: JL57 SFO → NRT, 11h 0m, 3,523 USD.

Recommendation: If you want the cheapest nonstop, go with ZIPAIR (634 USD). If you want the f